# Fraud Detection: Rule-based Analysis
Is notebook mein `data/flagged.csv` se rules ka performance aur fraud ke patterns dekhte hain. Notebook project root (`FraudDetection/`) se chalao, ya path adjust karo.

## 1. Data load karna
**Kya:** flagged data padhte hain (7 columns). **Kyun:** baaki saara analysis isi table par hai.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cols = ["trans_time", "amt", "is_fraud", "category", "state", "trans_hour", "flag_count"]
try:
    df = pd.read_csv("data/flagged.csv", usecols=cols, parse_dates=["trans_time"])
except FileNotFoundError:
    df = pd.read_csv("../data/flagged.csv", usecols=cols, parse_dates=["trans_time"])
df.head()

## 2. Overall numbers
**Kya:** total transactions, fraud count aur fraud rate. **Kyun:** fraud bahut kam hota hai (imbalanced data), isliye accuracy se zyada precision aur recall dekhte hain.

In [ ]:
total = len(df)
fraud = int(df["is_fraud"].sum())
print(f"Transactions: {total:,}")
print(f"Fraud: {fraud:,} ({fraud/total*100:.2f}%)")

## 3. Final rule ka performance
**Kya:** jo transaction 2 ya zyada rules se flag hui, uska precision aur recall. **Kyun:** precision batata hai flag kiye hue mein kitne asli fraud, recall batata hai kitne fraud pakde gaye.

In [ ]:
flagged = df["flag_count"] >= 2
caught = int((flagged & (df["is_fraud"] == 1)).sum())
precision = caught / flagged.sum() * 100
recall = caught / fraud * 100
print(f"Flagged: {flagged.sum():,} | Caught fraud: {caught:,}")
print(f"Precision: {precision:.1f}% | Recall: {recall:.1f}%")

## 4. Rules ki sankhya ke hisaab se trade-off
**Kya:** threshold 1, 2, 3 par precision/recall. **Kyun:** zyada rules maangne par precision badhta hai par recall girta hai.

In [ ]:
rows = []
for k in [1, 2, 3]:
    f = df["flag_count"] >= k
    c = int((f & (df["is_fraud"] == 1)).sum())
    rows.append({"min_rules": k, "flagged": int(f.sum()), "caught": c,
                 "precision_%": round(c / f.sum() * 100, 1) if f.sum() else 0,
                 "recall_%": round(c / fraud * 100, 1)})
pd.DataFrame(rows)

## 5. Fraud rate by hour
**Kya:** har ghante ka fraud rate. **Kyun:** night rule ko data se justify karne ke liye.

In [ ]:
by_hour = df.groupby("trans_hour")["is_fraud"].mean().mul(100)
by_hour.plot(kind="bar", figsize=(10, 4), title="Fraud rate by hour (%)")
plt.xlabel("Hour of day"); plt.ylabel("Fraud rate (%)")
plt.show()

## 6. Fraud rate by category
**Kya:** category-wise fraud rate. **Kyun:** kaun si categories zyada risky hain.

In [ ]:
by_cat = df.groupby("category")["is_fraud"].mean().mul(100).sort_values()
by_cat.plot(kind="barh", figsize=(8, 5), title="Fraud rate by category (%)")
plt.xlabel("Fraud rate (%)")
plt.show()

## 7. Monthly fraud trend
**Kya:** har mahine fraud transactions ki sankhya. **Kyun:** koi trend ya spike dekhne ke liye.

In [ ]:
monthly = df.groupby(df["trans_time"].dt.to_period("M").astype(str))["is_fraud"].sum()
monthly.plot(figsize=(10, 4), title="Fraud transactions per month")
plt.xlabel("Month"); plt.ylabel("Fraud count")
plt.xticks(rotation=90)
plt.show()

## 8. Insights aur recommendations

**Insights**
- Total 662,187 transactions mein 3,771 fraud hain (lagbhag 0.57%), yaani data bahut imbalanced hai.
- Final rule (2 ya zyada rules) ne 2,468 transactions flag kin, jinme 1,582 asli fraud the: **Precision 64.1%, Recall 42.0%**.
- Raat ke ghanton (0 se 3) mein fraud rate 1.4% se upar hai, baaki ghanton se kaafi zyada. Ye night rule ko support karta hai.
- Threshold 3 rules karne par precision lagbhag 100% ho jata hai par recall girkar 0.6% reh jata hai (dashboard ke 0 se 15 ghante wale filter par).

**Recommendations**
- Recall sirf 42.0% hai, yaani aadhe se zyada fraud nahi pakde. Naye rules jodne chahiye (jaise high-risk categories, raat ki transactions).
- Precision 64.1% theek hai, par har flag manual review mein jayega to threshold 2 par rakhna balanced hai.
- Raat ke ghanton ke liye alag, sakht rule ya extra verification (OTP) lagani chahiye.
- Rules ko hold-out data par test karna chahiye taaki overfitting na ho.